# Data Preprocessing

In [2]:
import pandas as pd

In [3]:
dataset=pd.read_csv('SeoulBikeData.csv',encoding='unicode_escape')

Handling Missing Values In Data

1->Checking Missing Values

In [4]:
dataset.isnull().sum()

Date                         0
Rented Bike Count            0
Hour                         0
Temperature(°C)              0
Humidity(%)                  0
Wind speed (m/s)             0
Visibility (10m)             0
Dew point temperature(°C)    0
Solar Radiation (MJ/m2)      0
Rainfall(mm)                 0
Snowfall (cm)                0
Seasons                      0
Holiday                      0
Functioning Day              0
dtype: int64

2->Handling Missing Values

In [ ]:
# Drop all null values in the dataset
dataset = dataset.dropna()

# Alternatively, fill missing values with the mean (for numerical columns)
dataset['Temperature(°C)'] = dataset['Temperature(°C)'].fillna(dataset['Temperature(°C)'].mean())

# Or use median or mode for imputation, depending on the column
dataset['Humidity(%)'] = dataset['Humidity(%)'].fillna(dataset['Humidity(%)'].median())

# If the column is categorical, fill missing values with the mode (most frequent value)
dataset['Seasons'] = dataset['Seasons'].fillna(dataset['Seasons'].mode()[0])


Handling Outliers

1->Detecting Outliers

In [6]:
dataset.count()

Date                         8760
Rented Bike Count            8760
Hour                         8760
Temperature(°C)              8760
Humidity(%)                  8760
Wind speed (m/s)             8760
Visibility (10m)             8760
Dew point temperature(°C)    8760
Solar Radiation (MJ/m2)      8760
Rainfall(mm)                 8760
Snowfall (cm)                8760
Seasons                      8760
Holiday                      8760
Functioning Day              8760
dtype: int64

In [7]:
from scipy.stats import zscore

# Calculate Z-scores for the numeric columns
dataset['z_score'] = zscore(dataset['Temperature(°C)'])



# Alternatively, using IQR to detect outliers:
Q1 = dataset['Temperature(°C)'].quantile(0.25)
Q3 = dataset['Temperature(°C)'].quantile(0.75)
IQR = Q3 - Q1


In [8]:
# Filter out rows with Z-score greater than 3 (considered as outliers)
df_no_outliers = dataset[dataset['z_score'].abs() <= 3]

# Filtering out rows outside the IQR range
df_no_outliers_iqr = dataset[(dataset['Temperature(°C)'] >= (Q1 - 1.5 * IQR)) & (dataset['Temperature(°C)'] <= (Q3 + 1.5 * IQR))]

In [10]:
df_no_outliers_iqr.count()

Date                         8760
Rented Bike Count            8760
Hour                         8760
Temperature(°C)              8760
Humidity(%)                  8760
Wind speed (m/s)             8760
Visibility (10m)             8760
Dew point temperature(°C)    8760
Solar Radiation (MJ/m2)      8760
Rainfall(mm)                 8760
Snowfall (cm)                8760
Seasons                      8760
Holiday                      8760
Functioning Day              8760
z_score                      8760
dtype: int64

In [9]:
df_no_outliers.count()

Date                         8760
Rented Bike Count            8760
Hour                         8760
Temperature(°C)              8760
Humidity(%)                  8760
Wind speed (m/s)             8760
Visibility (10m)             8760
Dew point temperature(°C)    8760
Solar Radiation (MJ/m2)      8760
Rainfall(mm)                 8760
Snowfall (cm)                8760
Seasons                      8760
Holiday                      8760
Functioning Day              8760
z_score                      8760
dtype: int64

Feature Engineering

•	Extracting parts of datetime (e.g., year, month, hour, day of the week).<br>
•	Creating interaction features (e.g., combining temperature with humidity).<br>
•	Encoding cyclical features (e.g., month, day of the week).

In [5]:
import numpy as np

In [12]:
# converting date columns to datetime data type
dataset['Date']=pd.to_datetime(dataset['Date'],dayfirst=True)

In [13]:
# Extract datetime components
#dataset['hour'] = dataset['Date'].dt.hour
dataset['day_of_week'] = dataset['Date'].dt.dayofweek
dataset['month'] = dataset['Date'].dt.month

# Create interaction features (e.g., temperature * humidity)
dataset['temp_humidity_interaction'] = dataset['Temperature(°C)'] * dataset['Humidity(%)']

# Encode cyclical features (e.g., months, days of the week)
dataset['month_sin'] = np.sin(2 * np.pi * dataset['month'] / 12)
dataset['month_cos'] = np.cos(2 * np.pi * dataset['month'] / 12)
dataset['day_of_week_sin'] = np.sin(2 * np.pi * dataset['day_of_week'] / 7)
dataset['day_of_week_cos'] = np.cos(2 * np.pi * dataset['day_of_week'] / 7)


In [14]:
dataset.head()

,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),...,Holiday,Functioning Day,z_score,day_of_week,month,temp_humidity_interaction,month_sin,month_cos,day_of_week_sin,day_of_week_cos
0,2017-12-01,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,...,No Holiday,Yes,-1.513957,4,12,-192.4,-2.449294e-16,1.0,-0.433884,-0.900969
1,2017-12-01,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,...,No Holiday,Yes,-1.539074,4,12,-209.0,-2.449294e-16,1.0,-0.433884,-0.900969
2,2017-12-01,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,...,No Holiday,Yes,-1.580936,4,12,-234.0,-2.449294e-16,1.0,-0.433884,-0.900969
3,2017-12-01,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,...,No Holiday,Yes,-1.597680,4,12,-248.0,-2.449294e-16,1.0,-0.433884,-0.900969
4,2017-12-01,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,...,No Holiday,Yes,-1.580936,4,12,-216.0,-2.449294e-16,1.0,-0.433884,-0.900969


Encoding Categorical Variables

Machine learning models require numerical inputs, so you need to convert categorical variables (e.g., season, weather type) into numerical representations.

•	Label Encoding: Converts categories into integer labels.<br>
•	One-Hot Encoding: Creates binary columns for each category (recommended for models that don’t work well with ordinal data).


In [15]:
# Label Encoding (for ordinal variables, e.g., 'season')
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
dataset['season_encoded'] = le.fit_transform(dataset['Seasons'])

# One-Hot Encoding (for non-ordinal categorical variables)
dataset = pd.get_dummies(dataset, columns=['Functioning Day', 'Holiday'], drop_first=True)


In [18]:
dataset.head()

,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),...,day_of_week,month,temp_humidity_interaction,month_sin,month_cos,day_of_week_sin,day_of_week_cos,season_encoded,Functioning Day_Yes,Holiday_No Holiday
0,2017-12-01,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,...,4,12,-192.4,-2.449294e-16,1.0,-0.433884,-0.900969,3,True,True
1,2017-12-01,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,...,4,12,-209.0,-2.449294e-16,1.0,-0.433884,-0.900969,3,True,True
2,2017-12-01,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,...,4,12,-234.0,-2.449294e-16,1.0,-0.433884,-0.900969,3,True,True
3,2017-12-01,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,...,4,12,-248.0,-2.449294e-16,1.0,-0.433884,-0.900969,3,True,True
4,2017-12-01,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,...,4,12,-216.0,-2.449294e-16,1.0,-0.433884,-0.900969,3,True,True


•	LabelEncoder(): Converts categories into integer values.<br>
•	pd.get_dummies(): Creates one binary column per category. drop_first=True avoids multicollinearity by dropping the first dummy column.


Scaling or Normalizing Data

Many machine learning algorithms (e.g., distance-based models like KNN or models that rely on gradient descent like linear regression) perform better when the data is scaled or normalized. This step ensures that no feature dominates due to differences in scale.

•	Standardization: Rescales the data to have a mean of 0 and a standard deviation of 1.<br>
•	Min-Max Scaling: Rescales the data to a specific range, typically [0, 1].


In [19]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Standardization (Z-score)
scaler = StandardScaler()
dataset[['Rented Bike Count',	'Hour',	'Temperature(°C)', 'Humidity(%)',	'Wind speed (m/s)',	'Visibility (10m)',
         	'Dew point temperature(°C)',	'Solar Radiation (MJ/m2)',	
            'Rainfall(mm)',	'Snowfall (cm)']] = scaler.fit_transform(dataset[['Rented Bike Count',	'Hour',	'Temperature(°C)', 'Humidity(%)',	'Wind speed (m/s)',	'Visibility (10m)',
         	'Dew point temperature(°C)',	'Solar Radiation (MJ/m2)',	
            'Rainfall(mm)',	'Snowfall (cm)']])


In [20]:
dataset.head()

,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),...,day_of_week,month,temp_humidity_interaction,month_sin,month_cos,day_of_week_sin,day_of_week_cos,season_encoded,Functioning Day_Yes,Holiday_No Holiday
0,2017-12-01,-0.698650,-1.661325,-1.513957,-1.042483,0.458476,0.925871,-1.659605,-0.655132,-0.1318,...,4,12,-192.4,-2.449294e-16,1.0,-0.433884,-0.900969,3,True,True
1,2017-12-01,-0.776175,-1.516862,-1.539074,-0.993370,-0.892561,0.925871,-1.659605,-0.655132,-0.1318,...,4,12,-209.0,-2.449294e-16,1.0,-0.433884,-0.900969,3,True,True
2,2017-12-01,-0.824240,-1.372399,-1.580936,-0.944257,-0.699556,0.925871,-1.667262,-0.655132,-0.1318,...,4,12,-234.0,-2.449294e-16,1.0,-0.433884,-0.900969,3,True,True
3,2017-12-01,-0.926571,-1.227936,-1.597680,-0.895144,-0.796059,0.925871,-1.659605,-0.655132,-0.1318,...,4,12,-248.0,-2.449294e-16,1.0,-0.433884,-0.900969,3,True,True
4,2017-12-01,-0.971535,-1.083473,-1.580936,-1.091596,0.554978,0.925871,-1.736177,-0.655132,-0.1318,...,4,12,-216.0,-2.449294e-16,1.0,-0.433884,-0.900969,3,True,True


In [ ]:
# Min-Max Scaling
scaler = MinMaxScaler()
dataset[['Rented Bike Count',	'Hour',	'Temperature(°C)', 'Humidity(%)',	'Wind speed (m/s)',	'Visibility (10m)',
         	'Dew point temperature(°C)',	'Solar Radiation (MJ/m2)',	
            'Rainfall(mm)',	'Snowfall (cm)']] = scaler.fit_transform(dataset[[
                'Rented Bike Count',	'Hour',	'Temperature(°C)', 'Humidity(%)',	'Wind speed (m/s)',	'Visibility (10m)',
         	'Dew point temperature(°C)',	'Solar Radiation (MJ/m2)',	
            'Rainfall(mm)',	'Snowfall (cm)']])

•	StandardScaler: Subtracts the mean and divides by the standard deviation to standardize features.<br>
•	MinMaxScaler: Rescales features to a specific range, typically [0, 1], which is useful when the data does not follow a Gaussian distribution.
